# Computing Reading Measures from Eye-Tracking Data

This tutorial demonstrates how to compute word-level (AOI based) reading measures from eye-tracking data using predefined Areas of Interest (AOIs).

The `measure_reading` method computes a set of reading measures for a dataset containing eye movement events together with an AOI definition for the presented text.

## What you will learn

In this tutorial, you will learn how to:

- create a simple dummy dataset using the `Dataset` class,
- define Areas of Interest (AOIs) for the text,
- compute word-level reading measures using `measure_reading`, and
- inspect the resulting DataFrame containing the computed measures.

In [ ]:
from pathlib import Path

import polars as pl
import pytest

from pymovements import Dataset
from pymovements import DatasetDefinition
from pymovements import Events
from pymovements import Gaze

In [ ]:
tmp_path = 'data/ReadingMeasures/'
definition = DatasetDefinition(name='dummy')
dataset = Dataset(definition, path=tmp_path)

# We need 'subject_id' and 'text_id' in trial_columns for measure_reading to work
fixation_data = pl.DataFrame({
    'name': ['fixation', 'fixation', 'fixation', 'fixation', 'fixation'],
    'onset': [0, 60, 200, 400, 600],
    'offset': [50, 100, 300, 500, 700],
    'duration': [50, 40, 100, 100, 100],
    'location_x': [100, 95, 140, 200, 10000],
    'location_y': [50, 45, 50, 50, 50],
    'subject_id': [5, 5, 5, 5, 5],
    'text_id': ['b0', 'b0', 'b0', 'b0', 'b0'],
})
events = Events(fixation_data, trial_columns=['subject_id', 'text_id'])
dataset.gaze = [Gaze(events=events)]

In [ ]:
events

In [ ]:
aoi_path = 'tests/files/potec_word_aoi_b0.tsv'
aoi_dict = {'b0': aoi_path}
aoi_dict

In [ ]:
pl.read_csv(aoi_path, separator='\t').head()

- compute the reading measures per word

In [ ]:
reading_measures = dataset.measure_reading(aoi_dict)

In [ ]:
reading_measures.frame.head()

### Description of the fields returned by `dataset.measure_reading(aoi_dict)`

The resulting DataFrame contains one row per word (Area of Interest, AOI) and subject, along with a range of eye-tracking reading measures.

| Column | Description |
|--------|-------------|
| `word` | The word (AOI) within the text. |
| `word_index` | Zero-based index of the word within the text. |
| `FFD` | **First Fixation Duration** — duration of the first fixation during the first pass only. |
| `SFD` | **Single Fixation Duration** — fixation duration when a word receives exactly one fixation; `0` if the word receives multiple fixations. |
| `FD` | **Fixation Duration** — duration of the first fixation on the word. |
| `FPRT` | **First Pass Reading Time** — sum of all fixation durations on the word during the first pass. |
| `FRT` | **First Reading Time** — total dwell time from first entering the word until first leaving it. |
| `TFT` | **Total Fixation Time** — total fixation time on the word (`FPRT + RRT`). |
| `RRT` | **Rereading Time** — sum of fixation durations occurring after the first pass. |
| `RPD_inc` | **Regression-Path Duration (inclusive)** — sum of all fixation durations from first entering the word until the first fixation to the right of the word, including fixations on the word itself. |
| `RPD_exc` | **Regression-Path Duration (exclusive)** — time spent on regressed words only, excluding fixations on the current word. |
| `RBRT` | **Right-Bounded Reading Time** — sum of fixation durations on the word before any word to its right is fixated. |
| `Fix` | **Fixation Indicator** — `1` if the word was fixated (`TFT > 0`). |
| `FPF` | **First Pass Fixation Indicator** — `1` if the word was fixated during the first pass (`FFD > 0`). |
| `RR` | **Rereading Indicator** — `1` if the word was reread (`RRT > 0`). |
| `FPReg` | **Regression Indicator** — `1` if regressions occurred (`RPD_exc > 0`). |
| `TRC_out` | **Total Regression Count (Outgoing)** — number of regressions originating from the word. |
| `TRC_in` | **Total Regression Count (Ingoing)** — number of regressions landing on the word. |
| `SL_in` | **Saccade Length (Ingoing)** — word distance between the current word and the previously fixated word at the time of the first fixation on the current word. |
| `SL_out` | **Saccade Length (Outgoing)** — word distance from the current word to the next fixated word, measured at the last fixation of the first reading pass. |
| `TFC` | **Total Fixation Count** — total number of fixations on the word. |
| `subject_id` | Identifier of the subject. |
| `text_id` | Identifier of the text. |